In [0]:
%restart_python
%pip install boto3
import boto3
import os
from botocore.exceptions import NoCredentialsError
import datetime
import sys
sys.path.insert(0, '/Workspace/Shared')
import etl_helpers
from pyspark.sql.functions import lit, col

tablename = "factrequestdivisionalstages"
runcycleid = etl_helpers.start_run_cycle(tablename)

os.makedirs("/dbfs/foi/dataload", exist_ok=True)  # make sure directory exists

try:
    df_lastrun = spark.sql(f"SELECT runcyclestartat as createddate FROM dimruncycle WHERE packagename = \"{tablename}\" AND success = 't' ORDER BY runcycleid DESC LIMIT 1")
    
    if df_lastrun.count() > 0:
        lastruntime = df_lastrun.first().createddate.strftime("%Y-%m-%d %H:%M:%S")
    else:
        lastruntime = "2019-01-01 00:00:00"
    print(lastruntime)

    query = f"""
        select
            mrd.foiministrydivisionid,
            mrd.divisionid,
            pad.name as divisionname,
            mrd.stageid,
            pads.name as stagename,
            mrd.created_at,
            mrd.createdby,
            mrd.updated_at,
            mrd.updatedby,
            mrd.divisionduedate,
            mrd.divisionreceiveddate,
            mrd.eapproval,
            mrd.foiministryrequestversion_id,
            mrd.foiministryrequest_id,
            mr.axisrequestid as visualrequestfilenumber,
            p.name as publicbody
        from 
        (
            select *
            from foi_mod.foiministryrequestdivisions
            QUALIFY ROW_NUMBER() OVER (PARTITION BY foiministryrequest_id, divisionid ORDER BY foiministrydivisionid DESC) = 1
        ) as mrd
        inner join foi_mod.programareadivisions pad on mrd.divisionid = pad.divisionid and pad.isactive in ('t', 'true')
        inner join foi_mod.programareadivisionstages pads on mrd.stageid = pads.stageid and pads.isactive in ('t', 'true')
        inner join foi_mod.programareas p on p.programareaid = pad.programareaid
        left join foi_mod.foiministryrequests mr on mr.foiministryrequestid = mrd.foiministryrequest_id and mr.version = mrd.foiministryrequestversion_id
        WHERE mrd.created_at > CAST('{lastruntime}' AS TIMESTAMP)
        """

    print(query)

    df = spark.sql(query)
    df.show()

    if df.count() == 0:
        raise Exception("no changes for today")

    df_mapped = df.selectExpr(
        "foiministrydivisionid AS foirequestdivisionstageid",
        f"{runcycleid} as runcycleid",
        "try_cast(divisionid AS INT) AS divisionid",
        "try_cast(divisionname AS STRING) AS divisionname",
        "try_cast(stageid AS INT) AS stageid",
        "try_cast(stagename AS STRING) AS stagename",
        "try_cast(created_at AS DATE) AS createddate",
        "try_cast(createdby AS STRING) AS createdby",
        "try_cast(updated_at AS DATE) AS modifieddate",
        "try_cast(updatedby AS STRING) AS modifiedby",
        "try_cast(divisionduedate AS DATE) AS divisionstageduedate",
        "try_cast(divisionreceiveddate AS DATE) AS divisionstagereceiveddate",
        "try_cast(eapproval AS STRING) AS divisionstageeapproval",
        "try_cast(foiministryrequestversion_id AS INT) AS foirequestversionid",
        "try_cast(foiministryrequest_id AS INT) AS foiministryrequestid",
        "try_cast(visualrequestfilenumber AS STRING) AS visualrequestfilenumber",
        "try_cast(publicbody AS STRING) AS publicbody",
        "'Y' as activeflag"
    )
    df_mapped.show()

    table_path = f"hive_metastore.default.{tablename}"

    # check if target table exist
    if spark.catalog.tableExists(table_path):
        from delta.tables import DeltaTable
        delta_table = DeltaTable.forName(spark, table_path)
        delta_table.alias("target").merge(
            df_mapped.alias("source"),
            "target.foiministryrequestid = source.foiministryrequestid AND target.divisionid = source.divisionid"
        ).whenMatchedUpdate(
            condition = "target.activeflag = 'Y'",
            set = {
                "activeflag": lit("N"),
            }
        ).execute()

        print("Matched records deactivated.")

        df_mapped.write.format("delta").mode("append").saveAsTable(f"hive_metastore.default.{tablename}")
    else:
        # If target table doesn't exist, create the table using the mapped dataframe
        print(f"Table {table_path} does not exist. Creating it now.")
        df_mapped.write.format("delta").mode("overwrite").saveAsTable(table_path)
    etl_helpers.end_run_cycle(runcycleid, 't', tablename)
except NoCredentialsError:
    print("Credentials not available")
    etl_helpers.end_run_cycle(runcycleid, 'f', tablename, "Credentials not available")
    raise Exception("notebook failed") from e
except Exception as e:
    if (str(e) == "no changes for today"):
        print("here")
        etl_helpers.end_run_cycle(runcycleid, 't', tablename)
    else:
        print(f"An error occurred: {e}")    
        etl_helpers.end_run_cycle(runcycleid, 'f', tablename, f"An error occurred: {e}")
        raise Exception("notebook failed") from e